### Data Preparation

In [1]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torch import nn
import torch

In [2]:
dataset = datasets.CIFAR10(
    root="data",
    train=True,
    download=True
)

pixels = dataset.data
mean = pixels.mean(axis=(0,1,2)) / 255
std = pixels.std(axis=(0, 1, 2)) / 255.0
print(mean)
print(std) 

train_transform =  transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
])

test_transform =  transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        std=std,
        mean=mean
    ),
])

training_dataset = datasets.CIFAR10(
    root="data",
    train=True,
    download=True,
    transform=train_transform
)

testing_dataset = datasets.CIFAR10(
    root="data",
    train=False,
    download=True,
    transform=test_transform
)

[0.49139968 0.48215841 0.44653091]
[0.24703223 0.24348513 0.26158784]


In [3]:
batch_size = 128
training_dataloader = DataLoader(training_dataset, batch_size, shuffle=True)
testing_dataloader = DataLoader(testing_dataset, batch_size)


print(training_dataloader.dataset)
print(testing_dataloader.dataset)


Dataset CIFAR10
    Number of datapoints: 50000
    Root location: data
    Split: Train
    StandardTransform
Transform: Compose(
               RandomCrop(size=(32, 32), padding=4)
               RandomHorizontalFlip(p=0.5)
               ToTensor()
           )
Dataset CIFAR10
    Number of datapoints: 10000
    Root location: data
    Split: Test
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=[0.49139968 0.48215841 0.44653091], std=[0.24703223 0.24348513 0.26158784])
           )


### Building a CNN from scratch

In [4]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.testingStack = nn.Sequential(
            # 3x3 x 3 x 32 + 32
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),

            # 3x3 x 32 x 32 + 32
            nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            # 3x3 x 32 x 64 + 64
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # 3x3 x 64x 128 + 128
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),

            # 128 x 10 + 10
            nn.Linear(128,10)
        )

    def forward(self,x):
        logits = self.testingStack(x)
        return logits
    
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
model = NeuralNetwork().to(device)

model.train()

NeuralNetwork(
  (testingStack): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU()
    (10): AdaptiveAvgPool2d(output_size=1)
    (11): Flatten(start_dim=1, end_dim=-1)
    (12): Linear(in_features=128, out_features=10, bias=True)
  )
)

### Inspecting

In [5]:
model.train()

# getting 1 example and its label
X,y = training_dataloader.dataset[0]

# it from cpu to device
X = X.unsqueeze(0).to(next(model.parameters()).device)

In [6]:
# predict a single value
pred = model(X)
pred

tensor([[ 0.0954,  0.0395,  0.0433, -0.0748, -0.0521, -0.0532,  0.0191, -0.0721,
          0.0905,  0.0648]], device='mps:0', grad_fn=<LinearBackward0>)

In [7]:
# apply to get probabilty
prob= pred.softmax(dim=1)

In [8]:
# get the loss
-prob[0,6].log()

tensor(2.2956, device='mps:0', grad_fn=<NegBackward0>)

In [9]:
# using built in CrossEntrophyLoss
loss_fn = nn.CrossEntropyLoss()
target = torch.tensor([y], dtype=torch.long, device=pred.device)
loss = loss_fn(pred,target)
loss

tensor(2.2956, device='mps:0', grad_fn=<NllLossBackward0>)

In [10]:
loss.backward()

In [11]:
# inspecting last layer 
last_layer = model.testingStack[-1] 

#inspecting a single grad value
grad_value = last_layer.weight.grad[9][127]
bias_value = last_layer.bias.grad

print(grad_value)
print(bias_value)


tensor(5.9080e-05, device='mps:0')
tensor([ 0.1087,  0.1028,  0.1032,  0.0917,  0.0938,  0.0937, -0.8993,  0.0919,
         0.1082,  0.1054], device='mps:0')


In [12]:
#before optimizer step
print(last_layer.weight)
# print(last_layer.bias)

Parameter containing:
tensor([[ 0.0255,  0.0662,  0.0424,  ...,  0.0298, -0.0025,  0.0255],
        [-0.0686,  0.0108, -0.0511,  ..., -0.0016,  0.0077, -0.0583],
        [ 0.0858,  0.0247,  0.0701,  ..., -0.0023,  0.0097,  0.0537],
        ...,
        [-0.0103,  0.0876,  0.0078,  ..., -0.0106, -0.0521,  0.0715],
        [ 0.0367, -0.0096, -0.0096,  ...,  0.0083,  0.0820, -0.0881],
        [-0.0775, -0.0847, -0.0626,  ...,  0.0157, -0.0580,  0.0281]],
       device='mps:0', requires_grad=True)


In [13]:
#large lr is just to reflect the different between the before and after gradient
optimizer = torch.optim.SGD(model.parameters(), lr=1)

In [14]:
optimizer.step()

In [15]:
print(last_layer.weight)
print(last_layer.bias)

Parameter containing:
tensor([[ 0.0246,  0.0634,  0.0390,  ...,  0.0274, -0.0025,  0.0255],
        [-0.0695,  0.0081, -0.0544,  ..., -0.0038,  0.0076, -0.0584],
        [ 0.0849,  0.0220,  0.0668,  ..., -0.0045,  0.0097,  0.0537],
        ...,
        [-0.0110,  0.0852,  0.0049,  ..., -0.0126, -0.0521,  0.0715],
        [ 0.0358, -0.0125, -0.0131,  ...,  0.0060,  0.0820, -0.0882],
        [-0.0784, -0.0874, -0.0660,  ...,  0.0134, -0.0581,  0.0280]],
       device='mps:0', requires_grad=True)
Parameter containing:
tensor([-0.0337, -0.0676, -0.0667, -0.1533, -0.1627, -0.1419,  0.9100, -0.1737,
        -0.0222, -0.0392], device='mps:0', requires_grad=True)
